# 固定 λ 的 NSS 拟合

本单元接受 [c01 期限结构输入](c01_TermStructureInputs.ipynb)，在给定两个衰减参数时估计四个系数，输出曲线、残差和诊断。λ 的网格校准归后续 c03；本单元的计算函数不绘图。输入与输出保持上游数值尺度，不取 log、不指数还原。

## 模型及经济约束

$$A(x)=\frac{1-e^{-x}}{x},\quad B(x)=A(x)-e^{-x},$$
$$y(\tau)=\beta_0+\beta_1 A(\lambda_1\tau)+\beta_2 B(\lambda_1\tau)+\beta_3 B(\lambda_2\tau).$$

零期限采用 $A(0)=1,B(0)=0$；很小的 $x$ 使用泰勒展开避免相减损失。$\tau$ 单位为 c01 定义的自然剩余天数，λ 单位为天的倒数，且必须满足 $0<\lambda_2<\lambda_1$。第二个曲率分量衰减更慢。这是本研究采用的参数关系，不是 NSS 普遍必需的大小关系。

长期极限为 $\beta_0$，短端极限为 $\beta_0+\beta_1$。通过 `coefficient_constraints` 显式提供 `level_lower_bound` 和 `short_lower_bound`；省略或设为 None 表示无该下界。正价格假设可使用正 ε 作为下界；log 输入不能直接套用正值约束。$\beta_1,\beta_2,\beta_3$ 允许正负。这些端点约束不保证所有中间期限为正，也不构成商品期货的无套利模型。

给定 λ 后最小化 $\sum_i(y_i-\hat y_i)^2$。无约束时使用线性最小二乘；有约束时重参数化为 $\gamma=(\beta_0,\beta_0+\beta_1,\beta_2,\beta_3)$，矩阵列变为 $(1-A,A,B_1,B_2)$，用 BVLS 求带下界的凸最小二乘，而非事后裁剪系数。严格正值通过调用者给出的正下界实现，数值验收容差按输入单位解释。

分离线性 β 与非线性 λ 的方法见 [Banholzer 等（2024），第 3.2 节](https://www.tandfonline.com/doi/abs/10.1080/02331934.2024.2389242)；端点及衰减参数的解释见 [Gilli 等（2010），第 2 节](https://enricoschumann.net/COMISEF/wps031.pdf)。BVLS 的优化问题与停止条件以当前环境 `scipy.optimize.lsq_linear` 文档为准。

## 对外函数

- `nss_basis(maturity_days, lambda1=..., lambda2=...)`：返回四列基函数矩阵。
- `evaluate_nss_curve(maturity_days, beta=..., lambda1=..., lambda2=...)`：在明确期限上求值，β 可为四元素序列或具名字典。
- `fit_nss_curve(maturity_days, input_values, ...)`：拟合一条曲线，返回具名系数、拟合值、残差、SSE／MSE／RMSE 和求解诊断。
- `fit_nss_sequence(term_structure_df, ...)`：固定同一组 λ，逐点独立拟合完整输入长表，返回 `coefficients_df` 和 `fitted_df`。

不设置合约数量门槛。少于四个观测或秩不足也尝试求解；有限解可保留为 `ok`，但明确标记 `rank_deficient`，不能据此解释为四个参数均已识别。没有有效输入、求解异常、不收敛或约束不满足时记录 `failed`，不换参数重试。配置错误直接报错，数据或数值失败返回状态。迭代上限状态即使有临时系数也不作为成功解发布。

本单元规则及真实 demo 归属见 [研究 AGENTS](../../AGENTS.md) 与 [研究说明](../../README.md#期限结构输入)。

In [10]:
from pathlib import Path
import sys
project_markers = [".git", ".env", "config/settings.py"]
current_path = Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")


In [11]:
from __future__ import annotations
import numpy as np
import pandas as pd
import pyarrow as pa
from scipy.optimize import lsq_linear
from config.data_contracts import pandas_to_arrow, arrow_to_pandas
from R04_Research.a01_Methods.b05_TermStructureModeling.c01_TermStructureInputs import (
    TERM_STRUCTURE_INPUT_SCHEMA, TERM_STRUCTURE_COLUMN_LABELS, TERM_STRUCTURE_TIMEZONE, TERM_STRUCTURE_TERMINAL_RULE,
    SAMPLING_POSITION_SCHEMA, iter_term_structure_curves,
)

NSS_FITTING_VERSION = "0.1.1"
NSS_COEFFICIENT_NAMES = ("beta0", "beta1", "beta2", "beta3")
NSS_COEFFICIENT_SCHEMA = pa.schema([
    *SAMPLING_POSITION_SCHEMA,
    pa.field("available_at", pa.timestamp("us", tz=TERM_STRUCTURE_TIMEZONE), nullable=False),
    *[pa.field(name, pa.float64(), nullable=False) for name in ("lambda1", "lambda2")],
    *[pa.field(name, pa.float64()) for name in NSS_COEFFICIENT_NAMES],
    pa.field("short_level", pa.float64()),
    *[pa.field(name, pa.float64()) for name in ("level_lower_bound", "short_lower_bound", "sse", "mse", "rmse", "condition_number", "optimality")],
    *[pa.field(name, pa.int64(), nullable=False) for name in ("observation_count", "design_rank", "solver_iterations")],
    *[pa.field(name, pa.bool_(), nullable=False) for name in ("level_bound_active", "short_bound_active")],
    *[pa.field(name, pa.string(), nullable=False) for name in ("identification_status", "solver", "solver_status", "fit_status", "fit_reason")],
], metadata={b"method_name": b"nss_fitting", b"method_version": NSS_FITTING_VERSION.encode(), b"terminal_rule": TERM_STRUCTURE_TERMINAL_RULE.encode(),
    b"primary_key": b"sample_id", b"input_domain": b"upstream supplied; no internal transformation",
    b"lambda_unit": b"inverse natural day", b"lambda_relation": b"0 < lambda2 < lambda1"})
NSS_FITTED_SCHEMA = pa.schema([
    *TERM_STRUCTURE_INPUT_SCHEMA,
    pa.field("fitted_value", pa.float64()), pa.field("residual", pa.float64()),
    pa.field("fit_status", pa.string(), nullable=False), pa.field("fit_reason", pa.string(), nullable=False),
], metadata={b"method_name": b"nss_fitting", b"method_version": NSS_FITTING_VERSION.encode(), b"terminal_rule": TERM_STRUCTURE_TERMINAL_RULE.encode(),
    b"primary_key": b"sample_id,contract_code", b"residual_definition": b"input_value - fitted_value"})


In [12]:
def _nss_lambdas(lambda1, lambda2):
    '''多个对外函数共同使用的衰减参数契约。'''
    if isinstance(lambda1, (bool, np.bool_)) or isinstance(lambda2, (bool, np.bool_)):
        raise ValueError("lambda1、lambda2 必须是有限实数")
    lambda1, lambda2 = float(lambda1), float(lambda2)
    if not (np.isfinite(lambda1) and np.isfinite(lambda2) and 0 < lambda2 < lambda1):
        raise ValueError("衰减参数必须满足 0 < lambda2 < lambda1")
    return lambda1, lambda2


def nss_basis(maturity_days, *, lambda1: float, lambda2: float) -> np.ndarray:
    '''NSS 四列基函数；零期限用极限，小期限避免相减损失。'''
    lambda1, lambda2 = _nss_lambdas(lambda1, lambda2)
    maturities = np.asarray(maturity_days, dtype=np.float64)
    if maturities.ndim != 1 or not np.isfinite(maturities).all() or (maturities < 0).any():
        raise ValueError("maturity_days 必须是一维有限非负期限")
    loadings = []
    with np.errstate(over="raise", invalid="raise", divide="raise"):
        for decay in (lambda1, lambda2):
            x = maturities * decay
            a = np.empty_like(x)
            b = np.empty_like(x)
            small = x <= 1e-4
            xs = x[small]
            a[small] = 1 - xs/2 + xs**2/6 - xs**3/24 + xs**4/120
            b[small] = xs/2 - xs**2/3 + xs**3/8 - xs**4/30
            a[~small] = -np.expm1(-x[~small]) / x[~small]
            b[~small] = a[~small] - np.exp(-x[~small])
            loadings.append((a, b))
    return np.column_stack([np.ones_like(maturities), loadings[0][0], loadings[0][1], loadings[1][1]])


def evaluate_nss_curve(maturity_days, *, beta, lambda1: float, lambda2: float) -> np.ndarray:
    '''在给定期限求输入尺度的 NSS 曲线值，不取 log 或指数还原。'''
    coefficients = np.asarray([beta[name] for name in NSS_COEFFICIENT_NAMES] if isinstance(beta, dict) else beta, dtype=float)
    if coefficients.shape != (4,) or not np.isfinite(coefficients).all():
        raise ValueError("beta 必须包含四个有限 NSS 系数")
    with np.errstate(over="raise", invalid="raise"):
        values = nss_basis(maturity_days, lambda1=lambda1, lambda2=lambda2) @ coefficients
    if not np.isfinite(values).all():
        raise FloatingPointError("NSS 求值产生非有限数值")
    return values


## 单条曲线的约束最小二乘

`coefficient_constraints` 的两个下界都按输入值单位解释，可为任意有限实数；None 表示无约束。求解容差 `solver_tol` 是 BVLS 的停止条件，`constraint_tolerance` 是解的绝对可行性验收容差，不是正下界本身。下界活动性也按这一容差记录。

残差统一为 `input_value - fitted_value`，SSE 直接对残差平方求和，MSE 为 SSE／有效观测数，RMSE 为其平方根。误差没有额外惩罚或跨曲线汇总；实验和后续 c03 决定怎样汇总。秩和条件数针对原四列 NSS 矩阵；秩不足时条件数写为空并标记不可完整识别，避免把矩形矩阵的有限条件数当成四个系数均可识别。

In [13]:
def fit_nss_curve(maturity_days, input_values, *, lambda1: float, lambda2: float,
                  coefficient_constraints: dict | None = None, solver_tol: float = 1e-10,
                  max_iter: int = 100, constraint_tolerance: float = 1e-8) -> dict:
    '''给定 λ 拟合四个系数；数据或求解失败返回状态，不设置合约数量门槛。'''
    lambda1, lambda2 = _nss_lambdas(lambda1, lambda2)
    constraints = dict(coefficient_constraints or {})
    if set(constraints) - {"level_lower_bound", "short_lower_bound"}:
        raise ValueError("未知系数约束；支持 level_lower_bound、short_lower_bound")
    bounds = [constraints.get(name) for name in ("level_lower_bound", "short_lower_bound")]
    for bound in bounds:
        if bound is not None and (isinstance(bound, (bool, np.bool_)) or not np.isfinite(float(bound))):
            raise ValueError("系数下界必须是有限实数或 None")
    bounds = [None if bound is None else float(bound) for bound in bounds]
    if not np.isfinite(solver_tol) or solver_tol <= 0 or not np.isfinite(constraint_tolerance) or constraint_tolerance < 0:
        raise ValueError("solver_tol 必须为正；constraint_tolerance 必须非负")
    if isinstance(max_iter, bool) or not isinstance(max_iter, (int, np.integer)) or max_iter <= 0:
        raise ValueError("max_iter 必须为正整数")
    fit = dict(lambda1=lambda1, lambda2=lambda2, beta=None, fitted_values=None, residuals=None,
        level_lower_bound=bounds[0], short_lower_bound=bounds[1], sse=None, mse=None, rmse=None,
        condition_number=None, optimality=None, observation_count=0, design_rank=0, solver_iterations=0,
        level_bound_active=False, short_bound_active=False, identification_status="unavailable",
        solver="bvls" if any(bound is not None for bound in bounds) else "lstsq",
        solver_status="not_run", fit_status="failed", fit_reason="")
    try:
        maturities = np.asarray(maturity_days, dtype=float)
        values = np.asarray(input_values, dtype=float)
        if maturities.ndim != 1 or values.ndim != 1 or maturities.shape != values.shape:
            fit["fit_reason"] = "unaligned_input_arrays"
            return fit
        fit["observation_count"] = len(values)
        if not np.isfinite(values).all():
            fit["fit_reason"] = "nonfinite_input_value"
            return fit
        basis = nss_basis(maturities, lambda1=lambda1, lambda2=lambda2)
        # 空数组没有可评价的拟合目标；不对非空数组设置最低数量。
        if not values.size:
            fit["fit_reason"] = "no_usable_inputs"
            return fit
        singular_values = np.linalg.svd(basis, compute_uv=False)
        rank = int(np.linalg.matrix_rank(basis))
        fit["design_rank"] = rank
        fit["identification_status"] = "full_rank" if rank == 4 else "rank_deficient"
        if rank == 4:
            fit["condition_number"] = float(singular_values[0] / singular_values[-1])
        with np.errstate(over="raise", invalid="raise"):
            if fit["solver"] == "lstsq":
                beta = np.linalg.lstsq(basis, values, rcond=None)[0]
                fit["solver_status"] = "solved"
                fit["optimality"] = float(np.max(np.abs(basis.T @ (basis @ beta - values))))
            else:
                constrained_basis = np.column_stack([1 - basis[:, 1], basis[:, 1], basis[:, 2], basis[:, 3]])
                lower = np.array([bounds[0] if bounds[0] is not None else -np.inf,
                                  bounds[1] if bounds[1] is not None else -np.inf, -np.inf, -np.inf])
                solution = lsq_linear(constrained_basis, values, bounds=(lower, np.full(4, np.inf)),
                    method="bvls", lsq_solver="exact", tol=solver_tol, max_iter=max_iter)
                fit["solver_status"] = str(solution.status)
                fit["solver_iterations"] = int(solution.nit)
                fit["optimality"] = float(solution.optimality)
                if not solution.success:
                    fit["fit_reason"] = "solver_not_converged: " + str(solution.message)
                    return fit
                gamma = solution.x
                beta = np.array([gamma[0], gamma[1] - gamma[0], gamma[2], gamma[3]])
            if not np.isfinite(beta).all():
                fit["fit_reason"] = "nonfinite_coefficients"
                return fit
            endpoint_values = (float(beta[0]), float(beta[0] + beta[1]))
            for endpoint, bound in zip(endpoint_values, bounds):
                if bound is not None and endpoint < bound - constraint_tolerance:
                    fit["fit_reason"] = "coefficient_constraint_violated"
                    return fit
            fitted_values = evaluate_nss_curve(maturities, beta=beta, lambda1=lambda1, lambda2=lambda2)
            residuals = values - fitted_values
            sse = float(residuals @ residuals)
            if not np.isfinite(sse):
                fit["fit_reason"] = "nonfinite_fit_error"
                return fit
        fit.update(beta=dict(zip(NSS_COEFFICIENT_NAMES, map(float, beta))), fitted_values=fitted_values,
            residuals=residuals, sse=sse, mse=sse / len(values), rmse=float(np.sqrt(sse / len(values))),
            level_bound_active=bounds[0] is not None and abs(endpoint_values[0]-bounds[0]) <= constraint_tolerance,
            short_bound_active=bounds[1] is not None and abs(endpoint_values[1]-bounds[1]) <= constraint_tolerance,
            fit_status="ok", fit_reason="")
    except (ValueError, FloatingPointError, np.linalg.LinAlgError) as error:
        fit["fit_reason"] = type(error).__name__ + ": " + str(error)
    return fit


## 完整序列与逐合约状态

只将 c01 的 `input_status == 'ok'` 行送入拟合，按同一批行同时排列合约、期限和值。系数表每个采样点一行；拟合表保留输入长表的全部行：有效且拟合成功的行写拟合值与残差；上游失败行保留原状态和原因；曲线求解失败时有效行记录该次失败，不删行。

少于四个观测的曲线可以拟合但系数通常不唯一；输出不以样本数提前拒绝，不对病态条件数自设阈值。状态、秩、条件数和误差是不同信息，正式实验分别使用。

In [14]:
def fit_nss_sequence(term_structure_df: pd.DataFrame | pa.Table, *, lambda1: float, lambda2: float,
                     coefficient_constraints: dict | None = None, solver_tol: float = 1e-10,
                     max_iter: int = 100, constraint_tolerance: float = 1e-8,
                     progress_callback=None) -> dict:
    '''逐点独立拟合固定 λ；返回系数表及保留全部合约行的拟合表。'''
    lambda1, lambda2 = _nss_lambdas(lambda1, lambda2)
    # 配置在遍历之前验证，空输入也不会掩盖非法配置。
    fit_nss_curve([], [], lambda1=lambda1, lambda2=lambda2, coefficient_constraints=coefficient_constraints,
        solver_tol=solver_tol, max_iter=max_iter, constraint_tolerance=constraint_tolerance)
    inputs_df = term_structure_df.to_pandas() if isinstance(term_structure_df, pa.Table) else term_structure_df.copy()
    inputs_df = inputs_df.reset_index(drop=True)
    fitted_values = np.full(len(inputs_df), np.nan)
    residuals = np.full(len(inputs_df), np.nan)
    row_status = inputs_df["input_status"].to_numpy(dtype=object).copy()
    row_reason = inputs_df["input_reason"].to_numpy(dtype=object).copy()
    coefficient_rows = []
    total_curves = inputs_df["sample_id"].nunique()
    for curve_number, curve in enumerate(iter_term_structure_curves(inputs_df), start=1):
        fit = fit_nss_curve(curve["maturity_days"], curve["input_value"], lambda1=lambda1, lambda2=lambda2,
            coefficient_constraints=coefficient_constraints, solver_tol=solver_tol,
            max_iter=max_iter, constraint_tolerance=constraint_tolerance)
        record = {name: curve[name] for name in SAMPLING_POSITION_SCHEMA.names}
        record["available_at"] = curve["available_at"]
        record.update({name: fit[name] for name in NSS_COEFFICIENT_SCHEMA.names
            if name not in SAMPLING_POSITION_SCHEMA.names and name not in (*NSS_COEFFICIENT_NAMES, "short_level", "available_at")})
        record.update(fit["beta"] or {name: None for name in NSS_COEFFICIENT_NAMES})
        record["short_level"] = None if fit["beta"] is None else fit["beta"]["beta0"] + fit["beta"]["beta1"]
        coefficient_rows.append(record)
        usable_rows_df = curve["row_status_df"].loc[curve["row_status_df"]["input_status"].eq("ok")].sort_values(["maturity_days", "contract_code"])
        row_indices = usable_rows_df.index.to_numpy()
        row_status[row_indices] = fit["fit_status"]
        row_reason[row_indices] = fit["fit_reason"]
        if fit["fit_status"] == "ok":
            fitted_values[row_indices] = fit["fitted_values"]
            residuals[row_indices] = fit["residuals"]
        if progress_callback is not None:
            progress_callback(curve_number, total_curves, fit)
    coefficients_df = pd.DataFrame(coefficient_rows, columns=NSS_COEFFICIENT_SCHEMA.names)
    fitted_df = inputs_df[TERM_STRUCTURE_INPUT_SCHEMA.names].copy()
    fitted_df["fitted_value"] = fitted_values
    fitted_df["residual"] = residuals
    fitted_df["fit_status"] = row_status
    fitted_df["fit_reason"] = row_reason
    return dict(coefficients_df=arrow_to_pandas(pandas_to_arrow(coefficients_df, NSS_COEFFICIENT_SCHEMA), NSS_COEFFICIENT_SCHEMA),
                fitted_df=arrow_to_pandas(pandas_to_arrow(fitted_df, NSS_FITTED_SCHEMA), NSS_FITTED_SCHEMA))


In [15]:
NSS_COLUMN_LABELS = {**TERM_STRUCTURE_COLUMN_LABELS, **{name: f"{name}（{meaning}）" for name, meaning in {
    "available_at": "报价信息完整可得时刻", "lambda1": "第一衰减参数，每自然日", "lambda2": "第二衰减参数，每自然日",
    "beta0": "长期极限水平", "beta1": "短端相对长期偏移", "beta2": "第一曲率系数", "beta3": "第二曲率系数",
    "short_level": "零期限极限水平", "level_lower_bound": "长期水平下界", "short_lower_bound": "短端水平下界",
    "sse": "残差平方和", "mse": "残差均方", "rmse": "残差均方根", "condition_number": "基函数矩阵条件数",
    "optimality": "求解器最优性指标", "observation_count": "实际拟合观测数", "design_rank": "四列基函数矩阵秩",
    "solver_iterations": "求解迭代次数", "level_bound_active": "长期下界是否活动", "short_bound_active": "短端下界是否活动",
    "identification_status": "参数识别状态", "solver": "求解方法", "solver_status": "求解器状态",
    "fit_status": "拟合状态", "fit_reason": "拟合状态原因", "fitted_value": "输入尺度拟合值", "residual": "输入值减拟合值",
}.items()}}


## 默认真实小样本试算（不保存数据文件）

参数由同主题 `c03_NSSCalibration.yaml` 统一定义：RB、N=15、gk_money、all_contracts、log_interpolate、fixed。每分钟采样轴增量为主力合约 GK 波动率 σ 乘该品种全部合约成交额，采样单位为完整训练期平均分钟增量的 15 倍，测试期固定此单位；N 不表示固定每 15 分钟采样。完整请求训练 2010—2024，测试 2025—2026-09-30。当前 Notebook 仅计算每侧指定的少量完整采样点，默认 32 个训练点与 16 个测试点，保留每点全部真实合约。不读取旧 NSS demo 作为试算输入，不写 Parquet、CSV 或其他计算结果文件；表格和图形仅保留在 Notebook 内。全区间九份产物由 c03 的显式云入口计算及回收。

In [16]:
from R04_Research.a01_Methods.b05_TermStructureModeling.c01_TermStructureInputs import (
    load_nss_demo_config,load_term_structure_demo_source,build_contract_terminals,prepare_term_structure_inputs,
)

from IPython.display import display
demo_config_path=candidate_root/'R04_Research/a01_Methods/b05_TermStructureModeling/c03_NSSCalibration.yaml'
demo_config=load_nss_demo_config(demo_config_path)
demo_source=load_term_structure_demo_source(candidate_root,demo_config,trial=True)
demo_terminal_df=build_contract_terminals(demo_source['calendar_df'])
demo_inputs_df=prepare_term_structure_inputs(demo_source['sampled_df'],demo_terminal_df,value_col=demo_config['input']['value_col'])
assert len(demo_inputs_df)==len(demo_source['sampled_df'])
assert demo_inputs_df['sample_id'].nunique()<=demo_config['trial']['train_points']+demo_config['trial']['test_points']
print(f"真实小样本：{demo_inputs_df['sample_id'].nunique()} 点、{len(demo_inputs_df)} 合约行；计算结果不写数据文件")


真实小样本：48 点、576 合约行；计算结果不写数据文件


In [17]:
demo_first_curve=next(iter_term_structure_curves(demo_inputs_df))
demo_lambdas=demo_config['nss']['fixed_lambda']
demo_unconstrained=fit_nss_curve(demo_first_curve['maturity_days'],demo_first_curve['input_value'],**demo_lambdas)
if demo_unconstrained['fit_status']=='ok':
    np.testing.assert_allclose(nss_basis(demo_first_curve['maturity_days'],**demo_lambdas)@np.array([demo_unconstrained['beta'][name] for name in NSS_COEFFICIENT_NAMES]),demo_unconstrained['fitted_values'])
    np.testing.assert_allclose(evaluate_nss_curve([0],beta=demo_unconstrained['beta'],**demo_lambdas),demo_unconstrained['beta']['beta0']+demo_unconstrained['beta']['beta1'])
demo_sequence=fit_nss_sequence(demo_inputs_df,**demo_lambdas,**demo_config['nss']['fitting'])
demo_coefficients_df,demo_fitted_df=demo_sequence['coefficients_df'],demo_sequence['fitted_df']
assert len(demo_coefficients_df)==demo_inputs_df['sample_id'].nunique()
assert len(demo_fitted_df)==len(demo_inputs_df)
valid=demo_fitted_df['fit_status'].eq('ok')
np.testing.assert_allclose(demo_fitted_df.loc[valid,'residual'].astype(float),
    (demo_fitted_df.loc[valid,'input_value']-demo_fitted_df.loc[valid,'fitted_value']).astype(float))
display(demo_coefficients_df.tail(8).rename(columns=NSS_COLUMN_LABELS).rename_axis('row（真实试算系数行）'))
print('c02 固定 λ、无约束拟合、基函数及曲线求值已实际试算；状态：',demo_coefficients_df['fit_status'].value_counts().to_dict())
print('未保存计算数据文件')


c02 固定 λ、无约束拟合、基函数及曲线求值已实际试算；状态： {'ok': 48}
未保存计算数据文件


,sample_id（采样编号）,t（采样分界）,sample_side（采样侧）,sample_number（侧内采样编号）,source_minute_index（源分钟坐标）,sample_at（采样时刻）,trading_date（Session 归属的交易日）,bar_at（源分钟结束时刻）,main_contract_code（主力合约）,interpolation_fraction（分钟内位置比例）,available_at（报价信息完整可得时刻）,lambda1（第一衰减参数，每自然日）,lambda2（第二衰减参数，每自然日）,beta0（长期极限水平）,beta1（短端相对长期偏移）,beta2（第一曲率系数）,beta3（第二曲率系数）,short_level（零期限极限水平）,level_lower_bound（长期水平下界）,short_lower_bound（短端水平下界）,sse（残差平方和）,mse（残差均方）,rmse（残差均方根）,condition_number（基函数矩阵条件数）,optimality（求解器最优性指标）,observation_count（实际拟合观测数）,design_rank（四列基函数矩阵秩）,solver_iterations（求解迭代次数）,level_bound_active（长期下界是否活动）,short_bound_active（短端下界是否活动）,identification_status（参数识别状态）,solver（求解方法）,solver_status（求解器状态）,fit_status（拟合状态）,fit_reason（拟合状态原因）
row（真实试算系数行）,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
40,75615,2024-12-31 15:00:00+08:00,test,9,1134257,2025-01-02 14:17:17.676020+08:00,2025-01-02,2025-01-02 14:18:00+08:00,RB2505.XSGE,0.2946,2025-01-02 14:18:00+08:00,0.01,0.003,4894.196014,-1608.505142,-608.921785,-3509.129256,3285.690872,0.000001,0.000001,622.440512,51.870043,7.202086,1849.267306,0.0,12,4,0,False,False,full_rank,bvls,3,ok,
41,75616,2024-12-31 15:00:00+08:00,test,10,1134304,2025-01-02 21:04:30.712464+08:00,2025-01-03,2025-01-02 21:05:00+08:00,RB2505.XSGE,0.511874,2025-01-02 21:05:00+08:00,0.01,0.003,5468.738507,-2190.619979,-724.051831,-4908.362873,3278.118528,0.000001,0.000001,619.814097,51.651175,7.186875,1852.458294,0.0,12,4,0,False,False,full_rank,bvls,3,ok,
42,75617,2024-12-31 15:00:00+08:00,test,11,1134341,2025-01-02 21:41:30.828791+08:00,2025-01-03,2025-01-02 21:42:00+08:00,RB2505.XSGE,0.513813,2025-01-02 21:42:00+08:00,0.01,0.003,4490.240363,-1222.018669,-446.652988,-2624.915585,3268.221694,0.000001,0.000001,244.810117,20.400843,4.516729,1852.749116,0.0,12,4,0,False,False,full_rank,bvls,3,ok,
43,75618,2024-12-31 15:00:00+08:00,test,12,1134433,2025-01-03 09:13:41.978311+08:00,2025-01-03,2025-01-03 09:14:00+08:00,RB2505.XSGE,0.699639,2025-01-03 09:14:00+08:00,0.01,0.003,4566.341123,-1291.795794,-523.376968,-2766.818564,3274.545329,0.000001,0.000001,401.963742,33.496979,5.787657,1858.21641,0.0,12,4,0,False,False,full_rank,bvls,3,ok,
44,75619,2024-12-31 15:00:00+08:00,test,13,1134474,2025-01-03 09:54:46.011853+08:00,2025-01-03,2025-01-03 09:55:00+08:00,RB2505.XSGE,0.766864,2025-01-03 09:55:00+08:00,0.01,0.003,6269.958823,-3010.950199,-936.837867,-6919.644869,3259.008624,0.000001,0.000001,399.155625,33.262969,5.767406,1858.542399,0.0,12,4,0,False,False,full_rank,bvls,3,ok,
45,75620,2024-12-31 15:00:00+08:00,test,14,1134523,2025-01-03 10:58:24.675184+08:00,2025-01-03,2025-01-03 10:59:00+08:00,RB2505.XSGE,0.411253,2025-01-03 10:59:00+08:00,0.01,0.003,4896.172625,-1657.608577,-546.190146,-3645.046073,3238.564048,0.000001,0.000001,309.952381,25.829365,5.08226,1859.047963,0.0,12,4,0,False,False,full_rank,bvls,3,ok,
46,75621,2024-12-31 15:00:00+08:00,test,15,1134563,2025-01-03 13:38:01.181667+08:00,2025-01-03,2025-01-03 13:39:00+08:00,RB2505.XSGE,0.019694,2025-01-03 13:39:00+08:00,0.01,0.003,5256.374689,-2015.775222,-644.039941,-4544.886224,3240.599467,0.000001,0.000001,344.871429,28.739286,5.360903,1860.317749,0.0,12,4,0,False,False,full_rank,bvls,3,ok,
47,75622,2024-12-31 15:00:00+08:00,test,16,1134615,2025-01-03 14:30:50.286774+08:00,2025-01-03,2025-01-03 14:31:00+08:00,RB2505.XSGE,0.838113,2025-01-03 14:31:00+08:00,0.01,0.003,5731.300769,-2474.581063,-805.369046,-5636.102897,3256.719705,0.000001,0.000001,453.750316,37.812526,6.149189,1860.73856,0.0,12,4,0,False,False,full_rank,bvls,3,ok,
